## A2A Client to Interact with Our Server

In [11]:
%pip install a2a-sdk==0.3.8 httpx

Note: you may need to restart the kernel to use updated packages.


### Importing Libraries and Utilities

In [12]:
import logging

from typing import Any
from uuid import uuid4

import httpx

from a2a.client import A2ACardResolver, A2AClient
from a2a.types import (
    AgentCard,
    MessageSendParams,
    SendMessageRequest,
    SendStreamingMessageRequest,
)

### A2A Client Setup and Implementation

In [ ]:
import httpx

from a2a.client import ClientFactory
from a2a.client.card_resolver import A2ACardResolver
from a2a.client.client import ClientConfig
from a2a.client import create_text_message_object

from a2a.types import (
    AgentCard,
    TaskArtifactUpdateEvent,
    TaskStatusUpdateEvent,
    TransportProtocol,
)


base_url = "http://localhost:8080"


def extract_artifact_text(artifact) -> str:
    texts = []

    for part in artifact.parts:
        root = getattr(part, "root", part)
        text = getattr(root, "text", None)

        if text:
            texts.append(text)

    return "".join(texts)


timeout = httpx.Timeout(
    connect=10.0,
    read=120.0,
    write=10.0,
    pool=10.0,
)


async with httpx.AsyncClient(timeout=timeout) as httpx_client:
    resolver = A2ACardResolver(
        httpx_client=httpx_client,
        base_url=base_url,
    )

    agent_card: AgentCard = await resolver.get_agent_card()

    config = ClientConfig(
        httpx_client=httpx_client,
        supported_transports=[
            TransportProtocol.jsonrpc,
        ],
        streaming=True,
    )

    client = ClientFactory(config).create(agent_card)

    request = create_text_message_object(
        content="How is Gotham City doing today? Give me a detailed answer."
    )

    print("Svar:\n", flush=True)

    async for task, event in client.send_message(request):

        if isinstance(event, TaskArtifactUpdateEvent):
            text = extract_artifact_text(event.artifact)

            if text:
                print(text, end="", flush=True)

        elif isinstance(event, TaskStatusUpdateEvent):
            if event.final:
                print("\n\n[Stream completed]", flush=True)